# SA BankAccount Phase 2 — Before/After Evaluation
CPU only; three runs per scenario, 60 seconds per run. Same hardware session for both versions.
Source patch embedded; no input dataset required. Patch SHA-256: d4fa86ba70b43fd87133b99337e07bd0c68f5af15fc53bdf12056c96c6740449


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq git ca-certificates postgresql-16 postgresql-client-16 nodejs npm
mkdir -p /tmp/bank-phase2-tools
npm install --prefix /tmp/bank-phase2-tools/node-runtime --no-audit --no-fund node@24.22.0
/tmp/bank-phase2-tools/node-runtime/node_modules/node/bin/node --version


In [ ]:
import os, json, subprocess, tempfile, time, shutil, socket
from pathlib import Path
os.environ['PATH']='/tmp/bank-phase2-tools/node-runtime/node_modules/node/bin:'+os.environ['PATH']
BASELINE='c347c921d77ba533cfacd8e180feccef720be013'
import base64
patch=Path('/tmp/phase2-code.patch')
patch.write_bytes(base64.b64decode('...'))
patches=[patch]
ROOT=Path(tempfile.mkdtemp(prefix='sa-bank-phase2-',dir='/kaggle/working'))
OUT=Path('/kaggle/working')/('phase2-report-'+time.strftime('%Y%m%d-%H%M%S'))
(OUT/'results').mkdir(parents=True)
def run(args,cwd=ROOT): subprocess.run(args,cwd=cwd,check=True)
run(['git','clone','https://github.com/HUY-TD16/SA_BankAccount.git','.'])
run(['git','checkout','--detach',BASELINE])
run(['git','apply','--check',str(patches[0])])
run(['git','apply',str(patches[0])])
pgbin='/usr/lib/postgresql/16/bin/'
pgdata=tempfile.mkdtemp(prefix='bank-phase2-pg-')
run(['chown','postgres:postgres',pgdata])
run(['runuser','-u','postgres','--',pgbin+'initdb','-D',pgdata,'-A','trust','--encoding=UTF8','--locale=C'])
# Fail on a busy port rather than sharing an unknown database instance.
with socket.socket() as s: s.bind(('127.0.0.1',55432))
run(['runuser','-u','postgres','--',pgbin+'pg_ctl','-D',pgdata,'-l',pgdata+'/server.log','-o','-h 127.0.0.1 -p 55432','-w','start'])
run([pgbin+'createdb','-h','127.0.0.1','-p','55432','-U','postgres','bank_phase2_test'])
os.environ['DATABASE_URL']='postgresql://postgres@127.0.0.1:55432/bank_phase2_test'
os.environ['OUTPUT']=str(OUT/'results')
os.environ['ROUNDS']='3'
os.environ['DURATION']='60'
os.environ['MEASUREMENT_ENVIRONMENT']='Kaggle CPU'
os.environ['UV_THREADPOOL_SIZE']='4'
run(['npm','ci'])
run(['node','node_modules/prisma/build/index.js','generate'])
run(['node','node_modules/prisma/build/index.js','migrate','deploy'])
run(['npm','run','phase2:baseline'])
run(['node','scripts/phase2/verify.cjs'])


In [ ]:
try:
    run(['npm','run','phase2:benchmark'])
    run(['npm','run','phase2:report'])
    shutil.copy(ROOT/'docs/performance/phase2/README.md',OUT/'README.md')
    shutil.copy(ROOT/'docs/performance/phase2/phase2-kaggle.ipynb',OUT/'phase2-kaggle.ipynb')
    # Record the CPU quota and PostgreSQL settings alongside the regular metadata.
    limits={}
    for key in ['cpu.max','cpuset.cpus.effective','memory.max']:
        f=Path('/sys/fs/cgroup')/key
        limits[key]=f.read_text().strip() if f.exists() else 'unavailable'
    limits['lscpu']=subprocess.check_output(['lscpu'],text=True)
    limits['postgresSettings']=subprocess.check_output([pgbin+'psql',os.environ['DATABASE_URL'],'-Atc',"SELECT name||'='||setting FROM pg_settings WHERE name IN ('max_connections','shared_buffers','work_mem','fsync','synchronous_commit') ORDER BY name"],text=True)
    (OUT/'results'/'kaggle-limits.json').write_text(json.dumps(limits,indent=2))
    archive=shutil.make_archive(str(OUT),'zip',OUT)
    print('Kết quả Kaggle:',archive)
finally:
    run(['runuser','-u','postgres','--',pgbin+'pg_ctl','-D',pgdata,'-m','fast','-w','stop'])
